# Day 11 — Spark Structured Streaming Fundamentals

**Structured Streaming **is Spark's stream-processing engine. 
It allows us to process **incoming data incrementally **using familiar DataFrame and SQL operations.
Instead of loading an entire dataset each time, Spark can **process new data as it becomes available.**

### Batch V/s Streaming
````
**Batch processing	                Structured Streaming**
Processes a bounded dataset	        Processes a continuously growing dataset
Uses spark.read	                    Uses spark.readStream
Uses df.write	                    Uses df.writeStream
Executes a finite job	            Can run continuously or until available data is processed
````

Structured Streaming typically uses** micro-batches**, where Spark processes newly available data in small batches.
 when new data is available --> it create micro batches and processes the file in incremental way


Spark does not automatically launch another concurrent micro-batch for that same query after 10 seconds. The next batch starts when the current batch finishes, potentially causing processing lag.

### Trigger
Triggers control when Spark attempts to process available data.
Trigger	Meaning
availableNow=True	Processes available data and stops
processingTime="10 seconds"	Attempts a micro-batch approximately every 10 seconds
Default trigger	Processes data as quickly as possible using the engine's default scheduling behavior

.trigger(availableNow=True)

In [0]:
%sql
select * from  workspace.customer360_bronze.transactions_source;

select * from workspace.customer360_bronze.transactions_stream;

## Step 1- Create a source Delta table

In [0]:
%python

# Create DataFrame with sample data
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

schema = StructType([
    StructField('transactionId', IntegerType(), False),
    StructField('custId', IntegerType(), False),
    StructField('serviceType', StringType(), False),
    StructField('amount', IntegerType(), False)
])

transaction_data = (
    (1, 1001, "Wireless", 120.50),
    (2, 1002, "Fiber", 145.99),
    (3, 1001, "Internet", 75.00),
    (4, 1003, "Wireless", 85.25)
)

dfTransaction = spark.createDataFrame(transaction_data, schema)
dfTransaction.write.format("delta").mode("overwrite").saveAsTable("workspace.customer360_bronze.transactions_source")

## Step 2- Read the Delta table as a stream

In [0]:
dfStream = (
    spark.readStream
    .format("delta")
    .table(
        "workspace.customer360_bronze.transactions_source"
    )
)


## Step 3 — Write to a target Delta table

In [0]:
%python

checkpoint_path = (
    "/Volumes/workspace/customer360_bronze/ingestion_metadata/checkpoints/transactions_day11"
)

# Ingest into Bronze
query = (
    dfStream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(
        "workspace.customer360_bronze.transactions_stream"
    )
)

query.awaitTermination()

## Step 4- verify the result

In [0]:
%sql
SELECT *
FROM workspace.customer360_bronze.transactions_stream
ORDER BY transactionId;

## Step 5- Add new transactions

execute step 5 and then step 2 and step 3
so you will be adding 2 records to source table and write them to stream table. 

In [0]:
%python

new_transactionData = [
        (5, 1004, "Fiber", 110.00),
        (6, 1001, "Wireless", 95.00)
]

dfNewTransactions = spark.createDataFrame(new_transactionData, schema)
dfNewTransactions.write.format("delta").mode("append").saveAsTable("workspace.customer360_bronze.transactions_source")

## Mini Challenge

In [0]:
from pyspark.sql import functions as F

dfCitySummary = (
    spark.table(
        "workspace.customer360_bronze.transactions_stream"
    )
    .groupBy("custId")
    .agg(
        F.count("*").alias("transactionCount"),
        F.sum("amount").alias("totalAmount")
    ).
)

display(dfCitySummary)

### What is Spark Structured Streaming, and how does it work?
Spark Structured Streaming is a stream-processing engine that allows us to process incoming data incrementally using DataFrame and SQL APIs. It typically processes data in micro-batches, where each batch handles newly available records. We use checkpoints to track processing progress and support recovery if the job fails. Depending on the use case, we can run the stream continuously or use the AvailableNow trigger for scheduled incremental processing.

````
**Step	Action	                            Result**
1	    Create source Delta table	        4 transactions 
2	    Read source using readStream	    Streaming DataFrame created 
3	    Write using writeStream	Initial     4 records ingested
4	    Query target	                    Initial records verified
5	    Append 2 transactions to source	    Source now has 6 records
6	    Rerun Steps 2 and 3	                Target now has 6 records
````